# Forest Carbon Satellite Image Analysis & Model Upscaling
**Author:** Remilekun Eunice Atobatele  
**Target Position:** Doctoral Researcher (Ref. No. 11590) — GFZ German Research Centre for Geosciences  
**Project:** EUFOR-G3W / FLUXCOM-X Integration

---  
### Overview
This interactive notebook demonstrates a complete Earth Observation (EO) analytical pipeline:
1. **Multi-Spectral Satellite Analysis:** Computes vegetation greenness (NDVI), Gross Primary Productivity proxy (NIRv), canopy moisture (NDII), and detects forest disturbance/loss.
2. **Gridded Model Integration:** Upscales high-resolution metrics into climate-model-ready NetCDF grid cells while preserving sub-grid heterogeneity.

In [ ]:
# Step 1: Import core numerical and visualization libraries
import numpy as np
import matplotlib.pyplot as plt

print("Environment initialized successfully.")

## 1. Simulate Multi-Spectral Satellite Scene (Sentinel-2 L2A)
We load simulated Sentinel-2 surface reflectance bands:
- **Blue (B2 - 490 nm)** & **Green (B3 - 560 nm)**
- **Red (B4 - 665 nm)**: Chlorophyll absorption
- **NIR (B8 - 842 nm)**: Cell structure reflectance
- **SWIR (B11 - 1610 nm)**: Canopy moisture sensitivity

In [ ]:
def load_satellite_scene(height=400, width=400):
    np.random.seed(42)
    y, x = np.ogrid[:height, :width]
    terrain = np.sin(x / 50.0) * np.cos(y / 50.0)

    blue = 0.03 + 0.01 * np.random.rand(height, width)
    green = 0.06 + 0.02 * np.random.rand(height, width) + 0.02 * terrain
    red = 0.04 + 0.02 * np.random.rand(height, width)
    nir = 0.45 + 0.10 * np.random.rand(height, width) + 0.08 * terrain
    swir = 0.15 + 0.04 * np.random.rand(height, width)

    # Add river feature
    river_path = (width // 2) + (np.sin(y.flatten() / 30.0) * 40).astype(int)
    for row in range(height):
        col_c = int(river_path[row])
        col_start = max(0, col_c - 10)
        col_end = min(width, col_c + 10)
        blue[row, col_start:col_end] = 0.22
        green[row, col_start:col_end] = 0.18
        red[row, col_start:col_end] = 0.08
        nir[row, col_start:col_end] = 0.02
        swir[row, col_start:col_end] = 0.01

    # Add agricultural zone
    agri_mask = (y < 130) & (x < 150)
    red[agri_mask], green[agri_mask], nir[agri_mask], swir[agri_mask] = 0.12, 0.14, 0.28, 0.20

    # Add forest disturbance patch
    disturb_mask = (y > 240) & (y < 350) & (x > 230) & (x < 360)
    red[disturb_mask], nir[disturb_mask], swir[disturb_mask] = 0.18, 0.12, 0.32

    return {
        "Blue": np.clip(blue, 0, 1),
        "Green": np.clip(green, 0, 1),
        "Red": np.clip(red, 0, 1),
        "NIR": np.clip(nir, 0, 1),
        "SWIR": np.clip(swir, 0, 1),
    }

scene = load_satellite_scene()
print("Multi-spectral Sentinel-2 scene loaded successfully.")

## 2. Compute Spectral Indices & Render Visual Analysis
Run the cell below to calculate **NDVI**, **NIRv** (GPP Proxy), **NDII** (Moisture), and **Forest Disturbance Detection**.

In [ ]:
red, nir, green, swir, blue = scene["Red"], scene["NIR"], scene["Green"], scene["SWIR"], scene["Blue"]
eps = 1e-6

# Spectral Indices
ndvi = (nir - red) / (nir + red + eps)
nirv = ndvi * nir
ndii = (nir - swir) / (nir + swir + eps)
nbr = (nir - swir) / (nir + swir + eps)
disturbance = (nbr < 0.05) & (red > 0.10)

# Composites
rgb_true = np.clip(np.stack([red, green, blue], axis=-1) / 0.25, 0, 1)
rgb_false = np.clip(np.stack([nir, red, green], axis=-1) / 0.50, 0, 1)

# Render 6-Panel Interactive Graphic
fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=130)
fig.suptitle("Earth Observation Forest Carbon & Disturbance Pipeline\nCandidate: Remilekun Eunice Atobatele | GFZ Ref: 11590", fontsize=14, fontweight="bold", y=0.98)

axes[0, 0].imshow(rgb_true); axes[0, 0].set_title("(A) True-Color (RGB B4-B3-B2)"); axes[0, 0].axis("off")
axes[0, 1].imshow(rgb_false); axes[0, 1].set_title("(B) False-Color Infrared (CIR B8-B4-B3)\nVegetation = Red"); axes[0, 1].axis("off")

im_c = axes[0, 2].imshow(ndvi, cmap="YlGn", vmin=-0.1, vmax=0.9)
axes[0, 2].set_title("(C) NDVI (Canopy Greenness & Cover)"); axes[0, 2].axis("off")
fig.colorbar(im_c, ax=axes[0, 2], fraction=0.046, pad=0.04, label="NDVI")

im_d = axes[1, 0].imshow(nirv, cmap="viridis", vmin=0.0, vmax=0.5)
axes[1, 0].set_title("(D) NIRv (Photosynthetic Capacity / GPP Proxy)"); axes[1, 0].axis("off")
fig.colorbar(im_d, ax=axes[1, 0], fraction=0.046, pad=0.04, label="NIRv")

im_e = axes[1, 1].imshow(ndii, cmap="RdYlBu", vmin=-0.5, vmax=0.7)
axes[1, 1].set_title("(E) NDII (Canopy Moisture & Drought Stress)"); axes[1, 1].axis("off")
fig.colorbar(im_e, ax=axes[1, 1], fraction=0.046, pad=0.04, label="NDII")

disturb_map = np.zeros((ndvi.shape[0], ndvi.shape[1], 3))
disturb_map[ndvi > 0.4] = [0.2, 0.6, 0.3]
disturb_map[nir < 0.05] = [0.1, 0.2, 0.5]
disturb_map[disturbance] = [0.9, 0.1, 0.1]
axes[1, 2].imshow(disturb_map); axes[1, 2].set_title("(F) Disturbance Detection (Red = Forest Loss)", fontsize=11, fontweight="bold", color="darkred"); axes[1, 2].axis("off")

plt.tight_layout()
plt.show()